In [1]:
# import neccesary libraries
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from app.utils import load_data_csv
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

# load data

In [2]:
df = load_data_csv()
df.head()

,Message ID,Subject,Message,classification,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Message ID      33716 non-null  int64 
 1   Subject         33427 non-null  object
 2   Message         33345 non-null  object
 3   classification  33716 non-null  object
 4   Date            33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [4]:
df.isna().sum()

Message ID          0
Subject           289
Message           371
classification      0
Date                0
dtype: int64

In [5]:
df = df.dropna()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 33107 entries, 1 to 33715
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Message ID      33107 non-null  int64 
 1   Subject         33107 non-null  object
 2   Message         33107 non-null  object
 3   classification  33107 non-null  object
 4   Date            33107 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.5+ MB


In [7]:
df.Message.duplicated().sum()

np.int64(3525)

In [8]:
df = df.drop_duplicates()

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 33107 entries, 1 to 33715
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Message ID      33107 non-null  int64 
 1   Subject         33107 non-null  object
 2   Message         33107 non-null  object
 3   classification  33107 non-null  object
 4   Date            33107 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.5+ MB


In [10]:
import re 
import nltk
nltk.download('stopwords')
from  nltk.corpus import stopwords
from  nltk.stem.porter import PorterStemmer
stp = set(stopwords.words("english"))
def preprocessing_text(text):
    text = re.sub("[^a-zA-Z0-9]"," ",text)
    text = text.lower()
    text = text.split()
    ps = PorterStemmer()
    words = [ps.stem(word) for word in text if not word in set(stp)]
    review = " ".join(words)
    return review
x = df["Message"].apply(preprocessing_text).to_numpy()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zakaria\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
print(x)

['gari product high island larger block 1 2 commenc saturday 2 00 p 6 500 gross carlo expect 9 500 10 000 gross tomorrow vastar own 68 gross product georg x 3 6992 forward georg weissman hou ect 12 13 99 10 16 daren j farmer 12 10 99 10 38 carlo j rodriguez hou ect ect cc georg weissman hou ect ect melissa grave hou ect ect subject vastar resourc inc carlo pleas call linda get everyth set go estim 4 500 come tomorrow 2 000 increas follow day base convers bill fischer bmar forward daren j farmer hou ect 12 10 99 10 34 enron north america corp georg weissman 12 10 99 10 00 daren j farmer hou ect ect cc gari bryan hou ect ect melissa grave hou ect ect subject vastar resourc inc darren attach appear nomin vastar resourc inc high island larger block 1 2 previous erron refer 1 well vastar expect well commenc product sometim tomorrow told linda harri get telephon number ga control provid notif turn tomorrow linda number record 281 584 3359 voic 713 312 1689 fax would pleas see someon contact 

In [12]:
y = df.loc[:,"classification"]

In [13]:
y = y.to_numpy()

In [14]:
y

array(['ham', 'ham', 'ham', ..., 'spam', 'spam', 'spam'],
      shape=(33107,), dtype=object)

In [15]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=0)

In [16]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((26485,), (6622,), (26485,), (6622,))

In [17]:
# bag of words
from sklearn.feature_extraction.text import TfidfVectorizer
vect = TfidfVectorizer(max_features=2000)
x_train_vect = vect.fit_transform(x_train).toarray()
x_test_vect = vect.transform(x_test).toarray()

In [18]:
x_train_vect.shape,x_test_vect.shape

((26485, 2000), (6622, 2000))

In [19]:
# label encoding 
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
y_train_enc = enc.fit_transform(y_train)
y_test_enc = enc.transform(y_test)

In [20]:
y_train_enc.shape,y_test_enc.shape

((26485,), (6622,))

In [21]:
model = GaussianNB()
model.fit(x_train_vect,y_train_enc)

,priors,None
,var_smoothing,1e-09


In [22]:
y_pred = model.predict(x_test_vect)
y_pred

array([1, 1, 0, ..., 1, 0, 0], shape=(6622,))

In [23]:
# confusion matrix
from sklearn.metrics import confusion_matrix,accuracy_score
cm = confusion_matrix(y_pred,y_test_enc)
cm

array([[3180,  152],
       [  86, 3204]])

In [24]:
acc = accuracy_score(y_pred=y_pred,y_true=y_test_enc)
print(f"the accuracy is:{acc*100:.2f}%")

the accuracy is:96.41%


In [25]:
from sklearn.metrics import classification_report
print(classification_report(y_pred,y_test_enc))

              precision    recall  f1-score   support

           0       0.97      0.95      0.96      3332
           1       0.95      0.97      0.96      3290

    accuracy                           0.96      6622
   macro avg       0.96      0.96      0.96      6622
weighted avg       0.96      0.96      0.96      6622



In [26]:
import joblib
joblib.dump(vect,"C:\\Users\\ASUS\\OneDrive\\Documents\\VS Code projects\\backend_email_spam_detection_project\\app\\model\\tfidf.joblib")
joblib.dump(model,"C:\\Users\\ASUS\\OneDrive\\Documents\\VS Code projects\\backend_email_spam_detection_project\\app\\model\\model.joblib")

['C:\\Users\\ASUS\\OneDrive\\Documents\\VS Code projects\\backend_email_spam_detection_project\\app\\model\\model.joblib']

In [27]:
from  collections import Counter
encoded_label = Counter(y_train_enc)
decoded_label = Counter(y_train)

In [28]:
encoded_label

Counter({np.int64(1): 13258, np.int64(0): 13227})

In [29]:
decoded_label

Counter({'spam': 13258, 'ham': 13227})

# conclusion
spam = 1
ham = 0